# Milestone 1 lab, notebook 6 of 8: Read the table first

## Where we are

| Notebook | What you do |
|---|---|
| 1. The question and five green checks | Meet Lotus Retails and the Monday question. Run the given pipeline for one night. Read its five checks |
| 2. Airflow runs it | Let Airflow run the same night. Read the DAG: its tasks and its arrows |
| 3. The number by hand | Add up one store's receipts yourself. Find out why the two numbers differ |
| 4. Who is hurt | For each table: who uses it, and what happens to them when it is wrong |
| 5. The check card | Meet the page you fill in before any check. Say what one row means |
| **6. Read the table first** (you are here) | Count the rows, find the smallest and largest values, count the empty cells |
| 7. Six questions for any table | Sort your worries under six questions. See which ones the five checks answer |
| 8. The production version | The same logic as a tested program |

**About ten minutes.**

In [1]:
import pandas as pd

from lab_helpers import sql

## Step 1: Look before you check

In notebook 3 you found the bad refund by luck. You happened to pick store `S03`.

There is a way to find such things on purpose. Before you write a check on a table, **read the table**. Ask three plain questions:

1. How many rows are there, per store and per day?
2. What are the smallest and the largest values?
3. Which cells are empty?

Every surprise in the answers is a candidate for a check.

## Step 2: Rows per store per day

The last eight days, one column per store.

In [2]:
rows = sql("""
    SELECT sales_date, store_id, count(*) AS lines
    FROM clean.sales_lines
    WHERE sales_date >= '2026-08-17'
    GROUP BY sales_date, store_id
    ORDER BY sales_date, store_id
""")
rows.pivot(index="sales_date", columns="store_id", values="lines")

store_id,S01,S02,S03,S04,S05,S06
sales_date,,,,,,
2026-08-17,130,99,117,112,70,81
2026-08-18,130,119,96,107,73,76
2026-08-19,127,118,113,105,74,90
2026-08-20,127,125,121,101,84,118
2026-08-21,191,126,147,134,86,104
2026-08-22,186,153,145,162,114,140
2026-08-23,126,121,118,111,105,96
2026-08-24,117,125,113,89,68,98


What do you see?

- **No hole.** Every store has a number on every day.
- **A rhythm.** Friday 21 and Saturday 22 August are the highest rows. A store's Tuesday is not like its Saturday.
- **Stores differ.** `S05` is always the smallest.

So there is no single "right" number of rows. A check that says "at least 100 lines" would fail `S05` every weekday. *Candidate check: compare each store's day with a normal day for that store and that weekday.*

## Step 3: The smallest and the largest

In [3]:
sql("""
    SELECT min(quantity)     AS smallest_quantity,
           max(quantity)     AS largest_quantity,
           min(amount_cents) AS smallest_amount,
           max(amount_cents) AS largest_amount,
           min(sold_at)      AS first_sale,
           max(sold_at)      AS last_sale
    FROM clean.sales_lines
""")

,smallest_quantity,largest_quantity,smallest_amount,largest_amount,first_sale,last_sale
0,-3,3,-2390,3585,2026-07-27 07:27:32,2026-08-24 21:54:59


Quantities and amounts go below zero. Those are the refunds. Sales and refunds are mixed in one answer, so it says little.

Read them apart. This is the step most people skip.

In [4]:
sql("""
    SELECT line_type,
           count(*)          AS lines,
           min(amount_cents) AS smallest_amount,
           max(amount_cents) AS largest_amount
    FROM clean.sales_lines
    GROUP BY line_type
    ORDER BY line_type
""")

,line_type,lines,smallest_amount,largest_amount
0,refund,73,-2390,949
1,sale,20409,26,3585


**Look at the refund row.** The largest amount of a refund is **949**. A refund gives money back, so its largest amount should be below zero.

That is the row from notebook 3. This time you did not need luck, and you did not need to pick the right store. One query over the whole table showed it. Which rows are they?

In [5]:
sql("""
    SELECT line_id, store_id, sales_date, product_name, quantity, amount_cents
    FROM clean.sales_lines
    WHERE line_type = 'refund' AND amount_cents > 0
""")

,line_id,store_id,sales_date,product_name,quantity,amount_cents
0,S03-0824-000008,S03,2026-08-24,Olive oil 500ml,-1,949


One row, in five weeks of data. *Candidate check: a refund's amount is below zero.*

## Step 4: The empty cells

In [6]:
sql("""
    SELECT count(*)                               AS rows,
           count(*) - count(member_id)            AS empty_member_id,
           count(*) - count(refund_of_receipt_id) AS empty_refund_of,
           count(*) - count(product_name)         AS empty_product_name
    FROM clean.sales_lines
""")

,rows,empty_member_id,empty_refund_of,empty_product_name
0,20482,13293,20409,0


Two columns are mostly empty. Is that wrong?

- `member_id` is empty when the customer has no loyalty card. Normal. The check card already said: no check.
- `refund_of_receipt_id` should be empty on a sale and filled on a refund. "Empty in which rows" is the real question.

In [7]:
sql("""
    SELECT line_type, count(*) AS lines, count(refund_of_receipt_id) AS with_refund_of
    FROM clean.sales_lines
    GROUP BY line_type
    ORDER BY line_type
""")

,line_type,lines,with_refund_of
0,refund,73,73
1,sale,20409,0


Every refund names the receipt it refunds, and no sale does. Good. *Candidate check: a refund names its receipt.*

One more place where rows can go missing: the cleaning step joins every line to a product and a store. A line that finds no partner is dropped, with no error. Compare the two counts.

In [8]:
sql("""
    SELECT (SELECT count(*) FROM raw.sales_lines)   AS raw_rows,
           (SELECT count(*) FROM clean.sales_lines) AS clean_rows
""")

,raw_rows,clean_rows
0,20482,20482


Equal tonight. *Candidate check: no line is lost between raw and clean.*

## Step 5: The whole table at once

What you just did has a name: **profiling**. It is reading a table before you check it.

The production code has the same three questions as one function. It gives one row per column.

In [9]:
from lotus_pipeline import profile
from lotus_pipeline.warehouse import connect

with connect() as wh:
    whole_table = pd.DataFrame(profile.profile_columns(wh, "clean.sales_lines"))

whole_table[["column", "empty_percent", "different_values", "smallest", "largest"]]

,column,empty_percent,different_values,smallest,largest
0,line_id,0.0,20482,S01-0727-000001,S06-0824-000098
1,receipt_id,0.0,6166,S01-T1-004001,S06-T2-004459
2,line_no,0.0,6,1,6
3,store_id,0.0,6,S01,S06
4,store_name,0.0,6,Lotus Ashby High Street,Lotus Fenwick Harbour
5,till_id,0.0,2,T1,T2
6,sold_at,0.0,6148,2026-07-27 07:27:32,2026-08-24 21:54:59
7,sales_date,0.0,29,2026-07-27,2026-08-24
8,product_id,0.0,40,P-001,P-040
9,product_name,0.0,40,Apples,Whole milk 2L


Profiling and checking are opposites, and you need both.

| | Profiling | A check |
|---|---|---|
| When | Once, before you write checks | Every night |
| It answers | "What is in this table?" | "Is this rule broken?" |
| It finds | Questions you did not have | Only what you thought to ask |

## What you now know

- **Profiling** is reading a table before you check it: rows, smallest and largest values, empty cells.
- Read the groups apart. The refund only showed when sales and refunds were separated.
- "Empty" is not wrong by itself. Ask in which rows it is empty.
- Each surprise became a candidate check. You have four.

**Say it in an interview:** "I profile a table before I write its checks. On the sales lines, the smallest and largest amount looked fine in total. Split by line type, the largest refund was plus $9.49, and a refund should never be above zero. That one query found what I had first found by luck."

**The theory:** profiling is this project's word. The nearest chapter idea is "distribution" in [Data Observability & Monitoring](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/data-observability-and-monitoring), section "Five signals to watch".

Next: [notebook 7](07_six_questions_for_any_table.ipynb). Sort the candidates, and see what is still missing.